# 🌍💹 NOTEBOOK DE DORIAN LE STATISTICIEN 	🌍💹

## === GUIDE des fonctions du module local datalake === 

from DATALAKE.data import *

**data = data_download_gmd("country")** -> Telechargement des données liés à un pays via global_macro_dataset (BD publique)

**data = data_download_fred("indicator"** : str, "start" : str , "end" : str) -> Telechargement d'un indicateur via la FRED (BD publique)

**data_storing(data : dataframe, "nom_fichier" : str)** -> Range un dataframe "data" dans le DATALAKE en parquet, que vous venez de télécharger d'internet 

**data = import_parquet("file_name" : str)** -> importe un dataframe du DATALAKE dans votre file cible (notebook ou .py)

**which_parquet()** -> Vous renvoie une liste de l'ensemble des parquets dispo dans le DATALAKE

In [ ]:
# NOTEBOK DE Dorian

import numpy as np 
import pandas as pd
from DATALAKE.data import *

In [24]:
df = import_parquet('data_main')
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]

df['date'] = pd.to_datetime(df['year'], format='%Y')
df.set_index('date', inplace=True)

if 'year' in df.columns:
    df.drop(columns=['year'], inplace=True)

df_detrended = pd.DataFrame(index=df.index)

# Groupe 1 : Log-Différence (Croissance)
cols_growth = ['gdp_nominal', 'cpi', 'oil_price', 'export', 'import', 'taux_changes']
for col in cols_growth:
    if col in df.columns:
        df_detrended[f'{col}'] = np.log(df[col]).diff()

# Groupe 2 : Différence Simple (Variation de taux)
cols_rates = ['expected_inflation', 'taux_directeur', 'yield_perpetual']
for col in cols_rates:
    if col in df.columns:
        df_detrended[f'{col}'] = df[col].diff()

df_detrended = df_detrended.dropna()
